# Tarea B — Temperatura, top-p y entropía de la distribución de salida

**MMIA 6013 · Tarea de práctica para el solver del Taller 03 v2**

Un modelo de lenguaje produce, para el siguiente token, un vector de logits. Consideramos un vocabulario de seis tokens, t0 a t5, con estos logits:

| Token | t0 | t1 | t2 | t3 | t4 | t5 |
|---|---|---|---|---|---|---|
| Logit | 2.0 | 1.0 | 0.5 | 0.2 | -1.0 | -3.0 |

El notebook se organiza en cuatro partes: (1) softmax con temperatura y entropía en bits, (2) muestreo de núcleo (top-p) con p = 0.9, (3) comprobación empírica con 10 000 muestras y divergencia KL, y (4) una pregunta conceptual sobre los límites T → 0 y T → ∞. Todas las cifras se calculan en el notebook, ninguna se escribe a mano.

## Parte 1 — Softmax con temperatura

Implementamos en NumPy la función softmax con temperatura p_i = exp(z_i/T) / Σ_j exp(z_j/T) de forma **numéricamente estable**: antes de exponenciar se resta el máximo de los logits (z_i − max(z)); como la softmax es invariante a desplazamientos constantes, el resultado no cambia, pero se evitan desbordamientos. Calculamos la distribución para T = 0.5, T = 1 y T = 2, y la entropía en bits de cada una, H = −Σ p_i log₂ p_i. La tabla de la salida presenta las tres distribuciones y las tres entropías con cuatro decimales, junto con la verificación de que cada distribución suma 1.

In [1]:
import json
import numpy as np

# ------------------------------------------------------------------
# Datos del problema (Parte 1 — softmax con temperatura)
# Vocabulario de seis tokens t0..t5 con sus logits
# ------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperatures = [0.5, 1.0, 2.0]


def softmax_con_temperatura(z, T):
    """
    Softmax con temperatura: p_i = exp(z_i/T) / sum_j exp(z_j/T).

    Implementación numéricamente estable: se resta el máximo de los
    logits antes de exponenciar, de modo que el mayor exponente es 0
    y no ocurre overflow. Matemáticamente el resultado es idéntico.
    """
    z = np.asarray(z, dtype=float)
    z_shift = (z - np.max(z)) / float(T)  # resta del máximo -> estabilidad
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z)


def entropia_bits(p):
    """Entropía en bits: H = -sum p_i log2 p_i (p_i = 0 aporta 0)."""
    p = np.asarray(p, dtype=float)
    p_pos = p[p > 0]
    return float(-np.sum(p_pos * np.log2(p_pos)))


# ------------------------------------------------------------------
# Cálculo de las tres distribuciones, sus entropías y las sumas
# ------------------------------------------------------------------
distribuciones = {}
entropias = {}
sumas = {}

for T in temperatures:
    p = softmax_con_temperatura(logits, T)
    distribuciones[T] = p
    entropias[T] = entropia_bits(p)
    sumas[T] = float(np.sum(p))

# ------------------------------------------------------------------
# Tabla con cuatro decimales
# ------------------------------------------------------------------
col_w = 12
sep = "-" * (8 + col_w * len(temperatures))

lines = []
lines.append("Softmax con temperatura: distribuciones y entropia (bits)")
lines.append(sep)
lines.append("Token".ljust(8) + "".join(f"{'T=' + str(T):<{col_w}}" for T in temperatures))
lines.append(sep)
for i, tok in enumerate(tokens):
    row = tok.ljust(8)
    for T in temperatures:
        row += f"{distribuciones[T][i]:<{col_w}.4f}"
    lines.append(row)
lines.append(sep)
row = "H (bits)".ljust(8)
for T in temperatures:
    row += f"{entropias[T]:<{col_w}.4f}"
lines.append(row)
row = "Suma".ljust(8)
for T in temperatures:
    row += f"{sumas[T]:<{col_w}.4f}"
lines.append(row)
lines.append(sep)

tabla = "\n".join(lines)
print(tabla)

# ------------------------------------------------------------------
# Resultados en JSON (números como floats planos)
# ------------------------------------------------------------------
resultados = {
    "tokens": tokens,
    "logits": [float(x) for x in logits],
    "temperatures": [float(T) for T in temperatures],
    "distributions": {
        f"T={T}": {tok: float(distribuciones[T][i]) for i, tok in enumerate(tokens)}
        for T in temperatures
    },
    "entropies_bits": {f"T={T}": float(entropias[T]) for T in temperatures},
    "distribution_sums": {f"T={T}": float(sumas[T]) for T in temperatures},
}

with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

# Impresión de los mismos números calculados (precisión completa)
print("\nValores guardados en resultados.json:")
print(json.dumps(resultados, indent=2))


Softmax con temperatura: distribuciones y entropia (bits)
--------------------------------------------
Token   T=0.5       T=1.0       T=2.0       
--------------------------------------------
t0      0.8231      0.5516      0.3583      
t1      0.1114      0.2029      0.2173      
t2      0.0410      0.1231      0.1693      
t3      0.0225      0.0912      0.1457      
t4      0.0020      0.0275      0.0800      
t5      0.0000      0.0037      0.0294      
--------------------------------------------
H (bits)0.9147      1.7998      2.2888      
Suma    1.0000      1.0000      1.0000      
--------------------------------------------

Valores guardados en resultados.json:
{
  "tokens": [
    "t0",
    "t1",
    "t2",
    "t3",
    "t4",
    "t5"
  ],
  "logits": [
    2.0,
    1.0,
    0.5,
    0.2,
    -1.0,
    -3.0
  ],
  "temperatures": [
    0.5,
    1.0,
    2.0
  ],
  "distributions": {
    "T=0.5": {
      "t0": 0.8230654368889799,
      "t1": 0.11138979402363647,
      "t2": 

## Parte 2 — Muestreo de núcleo (top-p)

Partiendo de la distribución con T = 1 de la Parte 1, aplicamos top-p con p = 0.9: ordenamos los tokens por probabilidad descendente, acumulamos sus probabilidades en ese orden y conservamos el **conjunto mínimo** de tokens cuya probabilidad acumulada alcanza 0.9, incluyendo el token que cruza el umbral. Los tokens fuera del núcleo se descartan y las probabilidades de los supervivientes se renormalizan para sumar 1. La salida reporta el orden, las probabilidades acumuladas, qué tokens sobreviven y la distribución renormalizada.

In [2]:
import json
import numpy as np

# ------------------------------------------------------------------
# Subtarea s2 (Parte 2): top-p (nucleus sampling) con p = 0.9
# sobre la distribución softmax con T = 1 de la Parte 1.
# Se recalcula todo desde los logits originales (nada hardcodeado).
# ------------------------------------------------------------------

# Datos de la Parte 1 (mismos parámetros)
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
T = 1.0
p_threshold = 0.9

# ------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable
# ------------------------------------------------------------------
def softmax_with_temperature(z, temperature):
    z = np.asarray(z, dtype=float)
    z_scaled = z / temperature
    z_shifted = z_scaled - np.max(z_scaled)  # resta del máximo para estabilidad
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z)

probs = softmax_with_temperature(logits, T)

# ------------------------------------------------------------------
# Ordenar tokens por probabilidad descendente
# ------------------------------------------------------------------
order = np.argsort(-probs)                      # índices de mayor a menor prob.
sorted_tokens = [tokens[i] for i in order]
sorted_probs = probs[order]
cumulative = np.cumsum(sorted_probs)

# ------------------------------------------------------------------
# Top-p: conjunto MÍNIMO de tokens cuya acumulada alcanza p = 0.9,
# incluyendo el token que hace cruzar el umbral.
# ------------------------------------------------------------------
k = int(np.searchsorted(cumulative, p_threshold, side="left")) + 1
k = min(k, len(tokens))
survivor_idx = order[:k]
survivors = [tokens[i] for i in survivor_idx]
cum_at_k = float(cumulative[k - 1])

# ------------------------------------------------------------------
# Renormalizar las probabilidades de los supervivientes
# ------------------------------------------------------------------
survivor_probs = probs[survivor_idx]
renorm = survivor_probs / np.sum(survivor_probs)

full_dist = {tok: float(pr) for tok, pr in zip(tokens, probs)}
cum_dict = {sorted_tokens[j]: float(cumulative[j]) for j in range(len(tokens))}
renorm_dist = {tokens[i]: float(r) for i, r in zip(survivor_idx, renorm)}

results = {
    "tokens": tokens,
    "logits": [float(z) for z in logits],
    "temperature": float(T),
    "p_threshold": float(p_threshold),
    "softmax_T1_full_distribution": full_dist,
    "sorted_tokens_desc": sorted_tokens,
    "sorted_probs_desc": [float(x) for x in sorted_probs],
    "cumulative_probs_desc": cum_dict,
    "num_surviving_tokens": int(k),
    "surviving_tokens": survivors,
    "cumulative_prob_of_survivors": cum_at_k,
    "renormalized_distribution": renorm_dist,
    "renormalized_sum": float(np.sum(renorm)),
}

with open("resultados.json", "w") as f:
    json.dump(results, f, indent=2)

# ------------------------------------------------------------------
# Reporte en stdout (mismos números que en resultados.json)
# ------------------------------------------------------------------
print("=== Parte 2: muestreo de núcleo (top-p) con p = 0.9 ===")
print(f"\nLogits: {dict(zip(tokens, [float(z) for z in logits]))}")
print(f"Temperatura: T = {T}")

print("\nSoftmax con T = 1 (distribución completa):")
for tok in tokens:
    print(f"  {tok}: {full_dist[tok]:.6f}")

print("\nTokens ordenados por probabilidad descendente:")
print("  " + " > ".join(sorted_tokens))
print("Probabilidades acumuladas (en ese orden):")
for tok in sorted_tokens:
    print(f"  {tok}: {cum_dict[tok]:.6f}")

print(f"\nUmbral p = {p_threshold}")
print(f"Conjunto mínimo que alcanza 0.9 (incluye el token que cruza el umbral): "
      f"{k} tokens")
print(f"Tokens supervivientes: {survivors}")
print(f"Probabilidad acumulada del núcleo: {cum_at_k:.6f}")

print("\nDistribución renormalizada del núcleo:")
for tok in survivors:
    print(f"  {tok}: {renorm_dist[tok]:.6f}")
print(f"Suma de la distribución renormalizada: {float(np.sum(renorm)):.6f}")


=== Parte 2: muestreo de núcleo (top-p) con p = 0.9 ===

Logits: {'t0': 2.0, 't1': 1.0, 't2': 0.5, 't3': 0.2, 't4': -1.0, 't5': -3.0}
Temperatura: T = 1.0

Softmax con T = 1 (distribución completa):
  t0: 0.551623
  t1: 0.202931
  t2: 0.123084
  t3: 0.091183
  t4: 0.027464
  t5: 0.003717

Tokens ordenados por probabilidad descendente:
  t0 > t1 > t2 > t3 > t4 > t5
Probabilidades acumuladas (en ese orden):
  t0: 0.551623
  t1: 0.754553
  t2: 0.877637
  t3: 0.968820
  t4: 0.996283
  t5: 1.000000

Umbral p = 0.9
Conjunto mínimo que alcanza 0.9 (incluye el token que cruza el umbral): 4 tokens
Tokens supervivientes: ['t0', 't1', 't2', 't3']
Probabilidad acumulada del núcleo: 0.968820

Distribución renormalizada del núcleo:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
Suma de la distribución renormalizada: 1.000000


## Parte 3 — Comprobación empírica

Tomamos 10 000 muestras de la distribución renormalizada de la Parte 2 con `numpy.random.default_rng(0)` y comparamos las frecuencias observadas con las probabilidades teóricas en un gráfico de barras agrupadas. Además, calculamos la divergencia KL en bits de la distribución empírica respecto de la teórica, KL(q ‖ p) = Σ q_i log₂(q_i / p_i), sobre los tokens del núcleo. Como muestra la salida y el gráfico, las frecuencias observadas están muy cerca de las probabilidades teóricas y la KL resulta prácticamente cero, lo que confirma que el muestreo reproduce la distribución objetivo.

In [3]:
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ==================================================================
# Parte 2 (recomputada): softmax estable con T = 1 y top-p con p=0.9
# ==================================================================
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
T = 1.0
p_threshold = 0.9

# Softmax numéricamente estable: restar el máximo antes de exponenciar
z = logits / T
z_shift = z - np.max(z)
exp_z = np.exp(z_shift)
probs_full = exp_z / np.sum(exp_z)

# Top-p: ordenar descendente, conjunto mínimo con acumulada >= p, renormalizar
order = np.argsort(-probs_full)
sorted_tokens = [tokens[i] for i in order]
sorted_probs = probs_full[order]
cumulative = np.cumsum(sorted_probs)
k = int(np.argmax(cumulative >= p_threshold)) + 1  # tamaño mínimo del núcleo

nucleus_tokens = sorted_tokens[:k]
nucleus_probs = sorted_probs[:k]
renorm_sorted = nucleus_probs / np.sum(nucleus_probs)
renorm = {tok: float(pr) for tok, pr in zip(nucleus_tokens, renorm_sorted)}
excluded_tokens = sorted_tokens[k:]

# Probabilidades teóricas del núcleo (alineadas con nucleus_tokens)
p_theory = np.array([renorm[t] for t in nucleus_tokens], dtype=float)

# ==================================================================
# Parte 3: muestreo empírico (10 000 muestras, default_rng(0))
# ==================================================================
rng = np.random.default_rng(0)
n_samples = 10000
draws = rng.choice(len(nucleus_tokens), size=n_samples, p=p_theory)
counts = np.bincount(draws, minlength=len(nucleus_tokens))
q_emp = counts / n_samples

# KL en bits: KL(q || p) = sum_i q_i * log2(q_i / p_i), con 0*log(0) := 0
mask = q_emp > 0
kl_bits = float(np.sum(q_emp[mask] * np.log2(q_emp[mask] / p_theory[mask])))

# ==================================================================
# Gráfico de barras agrupadas: probabilidad teórica vs frecuencia empírica
# ==================================================================
x = np.arange(len(nucleus_tokens))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width / 2, p_theory, width,
       label="Probabilidad teórica (top-p renormalizada)", color="#1f77b4", alpha=0.9)
ax.bar(x + width / 2, q_emp, width,
       label=f"Frecuencia empírica ({n_samples} muestras)", color="#ff7f0e", alpha=0.9)
for xi, pt, qe in zip(x, p_theory, q_emp):
    ax.text(xi - width / 2, pt + 0.006, f"{pt:.4f}", ha="center", fontsize=8)
    ax.text(xi + width / 2, qe + 0.006, f"{qe:.4f}", ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(nucleus_tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_title(f"Top-p (T=1, p={p_threshold}): teórica vs empírica  |  KL = {kl_bits:.6f} bits")
ax.set_ylim(0, max(p_theory.max(), q_emp.max()) * 1.18)
ax.legend(loc="upper right")
ax.text(0.02, 0.90, f"Excluidos del núcleo: {', '.join(excluded_tokens)}",
        transform=ax.transAxes, fontsize=9, va="top", color="dimgray")
plt.tight_layout()
plt.savefig("top_p_teorico_vs_empirico.png", dpi=120)
plt.close(fig)

# ==================================================================
# Resultados
# ==================================================================
results = {
    "tokens": tokens,
    "logits": [float(v) for v in logits],
    "temperature": float(T),
    "p_threshold": float(p_threshold),
    "softmax_T1_full_distribution": {t: float(pr) for t, pr in zip(tokens, probs_full)},
    "sorted_tokens_desc": sorted_tokens,
    "sorted_probs_desc": [float(v) for v in sorted_probs],
    "cumulative_probs_desc": {t: float(c) for t, c in zip(sorted_tokens, cumulative)},
    "num_surviving_tokens": int(k),
    "surviving_tokens": nucleus_tokens,
    "excluded_tokens": excluded_tokens,
    "renormalized_distribution": renorm,
    "rng_seed": 0,
    "n_samples": int(n_samples),
    "observed_counts": {t: int(c) for t, c in zip(nucleus_tokens, counts)},
    "observed_frequencies": {t: float(q) for t, q in zip(nucleus_tokens, q_emp)},
    "theoretical_probs_nucleus": {t: float(pr) for t, pr in zip(nucleus_tokens, p_theory)},
    "kl_empirical_vs_theoretical_bits": kl_bits,
}

with open("resultados.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))


{
  "tokens": [
    "t0",
    "t1",
    "t2",
    "t3",
    "t4",
    "t5"
  ],
  "logits": [
    2.0,
    1.0,
    0.5,
    0.2,
    -1.0,
    -3.0
  ],
  "temperature": 1.0,
  "p_threshold": 0.9,
  "softmax_T1_full_distribution": {
    "t0": 0.5516226378718239,
    "t1": 0.2029306277578035,
    "t2": 0.12308364752983937,
    "t3": 0.09118260875807133,
    "t4": 0.02746367398498595,
    "t5": 0.003716804097476065
  },
  "sorted_tokens_desc": [
    "t0",
    "t1",
    "t2",
    "t3",
    "t4",
    "t5"
  ],
  "sorted_probs_desc": [
    0.5516226378718239,
    0.2029306277578035,
    0.12308364752983937,
    0.09118260875807133,
    0.02746367398498595,
    0.003716804097476065
  ],
  "cumulative_probs_desc": {
    "t0": 0.5516226378718239,
    "t1": 0.7545532656296273,
    "t2": 0.8776369131594667,
    "t3": 0.968819521917538,
    "t4": 0.9962831959025239,
    "t5": 1.0
  },
  "num_surviving_tokens": 4,
  "surviving_tokens": [
    "t0",
    "t1",
    "t2",
    "t3"
  ],
  "excluded_tok

## Parte 4 — Pregunta conceptual

**¿Qué le pasa a la distribución cuando T → 0?** Al dividir los logits entre una temperatura cada vez más pequeña, las diferencias entre logits se amplifican: el término exp(z_max/T) domina a todos los demás y la masa de probabilidad se concentra en el token de mayor logit (aquí t0). En el límite T → 0⁺ la distribución degenera en una delta: probabilidad 1 para el argmax y 0 para el resto (una distribución one-hot). Su entropía tiende al mínimo posible, 0 bits. Esto es coherente con la evidencia de la Parte 1: como muestra la tabla anterior, la entropía disminuye al bajar la temperatura — la de T = 0.5 es la menor de las tres, la de T = 1 es intermedia y la de T = 2 la mayor —, y esa tendencia decreciente extrapola hacia 0 bits en el límite.

**¿Qué le pasa cuando T → ∞?** Los logits divididos entre una temperatura enorme se vuelven prácticamente iguales (z_i/T → 0 para todo i), de modo que exp(z_i/T) → 1 para todos los tokens y la distribución tiende a la **uniforme**, con probabilidad 1/V para cada uno de los V tokens (1/6 aquí). La entropía tiende a su máximo posible, log₂ V = log₂ 6 bits, lo que también concuerda con el crecimiento monótono de la entropía observado en la Parte 1 al aumentar T.

**¿Por qué T → 0 equivale a la decodificación voraz (greedy)?** La decodificación greedy elige en cada paso el token con el mayor logit, es decir, el argmax de los logits. Como la softmax con cualquier T > 0 es una transformación **monótona** de los logits, el argmax de la distribución coincide siempre con el argmax de los logits, independientemente del valor de T. Cuando T → 0, toda la probabilidad se concentra en ese argmax, de modo que muestrear de la distribución devuelve siempre el mismo token que elegiría greedy: el muestreo se vuelve determinista y ambas estrategias coinciden. La entropía nula del límite refleja exactamente esa ausencia de aleatoriedad.

Nota práctica: evaluar la softmax con T muy pequeño puede causar desbordamiento numérico (de ahí restar el máximo antes de exponenciar en la implementación estable); en la práctica, T → 0 se implementa directamente con el argmax.